# 06 · Are the synergies driven by commands? (Q4, optional)

**Owner:** Person A · **Read first:** Lecture 9 (linear dynamical systems), reading 1008_02 (Gong & Saxena)

**Model:** treat the pose as a state `z` and the EMG as an input `u`:

    z(t+1) = A z(t) + B u(t)

Fit A and B with least squares (`np.linalg.lstsq`). Compare against **B = 0** (no EMG: the hand moves on
its own). **We expect:** the EMG model predicts much better when run forward for many steps.
Use train frames with valid pose, and never step across a gap or between recordings.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01 (data/ is never committed)

In [ ]:
# ── Load the shared data made by notebook 01 (runs 01 first if needed) ───────
import numpy as np
if not PROCESSED.exists():
    %run notebooks/experiments/01_data_prep/01_data_prep.ipynb
d = np.load(PROCESSED)

emg        = d["emg"]          # (N, 16)  EMG envelope, 50 samples/s, non-negative
pose       = d["pose"]         # (N, 20)  joint angles in degrees
pose_valid = d["pose_valid"]   # (N,)     False where pose tracking failed
rec        = d["rec"]          # (N,)     which recording (0–29) each row comes from
split      = d["split"]        # (N,)     0 = train, 1 = test, -1 = gap (use neither)
stage_of_rec, side_of_rec = d["stage"], d["side"]   # (30,) task name and hand per recording

train, test = split == 0, split == 1
SEED = 0   # pass random_state=SEED to every model so results repeat
print(f"{len(emg)} frames | {train.sum()} train | {test.sum()} test | {len(stage_of_rec)} recordings")

### The variables you start with
`N` = number of frames in all 30 recordings together (≈ 90,000: 30 recordings × ~60 s × 50 frames/s).
Every row of every array is one frame, 20 ms long. Boolean arrays are **masks**: `emg[train]` keeps only the train rows.

| Name | Shape | dtype | What it is |
|---|---|---|---|
| `emg` | (N, 16) | float32 | EMG envelope, one column per electrode, ≥ 0, roughly 0–1 |
| `pose` | (N, 20) | float32 | joint angles in **degrees**, one column per joint |
| `pose_valid` | (N,) | bool | False where pose tracking failed (don't use those `pose` rows) |
| `rec` | (N,) | int16 | which recording (0–29) the frame comes from |
| `split` | (N,) | int8 | 0 = train, 1 = test, −1 = gap between them (use neither) |
| `train`, `test` | (N,) | bool | `split == 0`, `split == 1` |
| `stage_of_rec` | (30,) | str | task name of each recording; per frame: `stage_of_rec[rec]` → (N,) |
| `side_of_rec` | (30,) | str | `"left"` or `"right"` per recording; per frame: `side_of_rec[rec]` → (N,) |

Letters used in the docstrings below: **T** = number of frames in the array you pass in (any subset of N),
**k** = number of synergies, **C** = number of gesture classes. Run a helper on a tiny made-up array first
(e.g. `np.random.rand(5, 16)`) to check it returns the shape its docstring promises.

## Before you run anything: your prediction

Write what you expect and why (2–3 sentences). Compare at the end. Surprises are the interesting part.

> *Prediction:*

## Task 1 · Build (z(t), u(t), z(t+1)) triples inside each recording's train block

In [ ]:
import numpy as np

def make_triples(pose: np.ndarray, emg: np.ndarray, rec: np.ndarray,
                 mask: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Collect every step t → t+1 where both frames are usable and in the same recording.

    Parameters
    ----------
    pose : (N, 20) float, joint angles (degrees): the state z
    emg  : (N, 16) float, EMG envelope: the input u
    rec  : (N,) int, recording of each frame
    mask : (N,) bool, frames allowed (e.g. train & pose_valid)

    Returns
    -------
    Z      : (M, 20) float, z(t)
    U      : (M, 16) float, u(t)
    Z_next : (M, 20) float, z(t+1)
    M = number of t where mask[t] and mask[t+1] are True and rec[t] == rec[t+1].
    """
    # TODO
    raise NotImplementedError


Z, U, Z_next = None, None, None   # TODO: make_triples(pose, emg, rec, train & pose_valid)

## Task 2 · Fit A and B (with EMG) and A alone (without EMG)

In [ ]:
def fit_AB(Z: np.ndarray, U: np.ndarray, Z_next: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Least-squares fit of z(t+1) = A z(t) + B u(t).

    Parameters
    ----------
    Z : (M, 20), U : (M, 16), Z_next : (M, 20), from make_triples

    Returns
    -------
    A : (20, 20), B : (20, 16)
    In the maths z is a column vector; in code each frame is a ROW, so the model reads
    Z_next ≈ Z @ A.T + U @ B.T. Stack [Z, U] side by side → (M, 36) and use np.linalg.lstsq.
    """
    # TODO
    raise NotImplementedError


def fit_A(Z: np.ndarray, Z_next: np.ndarray) -> np.ndarray:
    """Least-squares fit of z(t+1) = A z(t) (no EMG). Z, Z_next: (M, 20). Returns A: (20, 20)."""
    # TODO
    raise NotImplementedError


A_full, B_full = None, None   # TODO: fit_AB(Z, U, Z_next)
A_only = None                 # TODO: fit_A(Z, Z_next)

## Task 3 · Free rollout
Start from a real test pose and run each model forward 1, 5 and 25 steps, feeding it its own predictions
(and the real EMG for the input model). Also score "no change" (z stays the same). Lower error = better.

In [ ]:
def rollout(z0: np.ndarray, U_future: np.ndarray, A: np.ndarray, B: np.ndarray | None) -> np.ndarray:
    """Run the model forward, feeding it its own predictions.

    Parameters
    ----------
    z0       : (20,) float, the real starting pose
    U_future : (H, 16) float, the real EMG for the next H steps (ignored when B is None)
    A        : (20, 20)
    B        : (20, 16), or None for the model without EMG

    Returns
    -------
    Z_pred : (H, 20) float, predicted pose after 1, 2, …, H steps
    """
    # TODO
    raise NotImplementedError


HORIZONS = [1, 5, 25]
# Starting points: test frames t whose next 25 frames are all test & pose_valid and in the same recording.
starts = None   # (S,) int: TODO find them (make_triples-style logic, but 25 steps ahead)

errors: dict[str, dict[int, float]] = {}   # model → {horizon: mean angular error (°) at that step}
# TODO: for each start t: rollout from pose[t] with emg[t : t+25]; compare step h with pose[t + h]
# TODO: "no change" model: the prediction for every step is pose[t]
# TODO: errors["no change"], errors["A only"], errors["A and B"]; then fill the table below

## Result
| Model | Error after 1 step | 5 steps | 25 steps |
|---|---|---|---|
| No change | | | |
| Without EMG (A only) | | | |
| With EMG (A and B) | | | |